# Milestone 1 - Basic AI Model Training


## Environment Setup & Hardware Verification

Hardware detection and package installation for running locally or in Colab.

In [ ]:
# System and hardware verification - putting this in due to the numerous issues I have had on my local machine with this.
import sys
import platform
import os

print(f"Python version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")

try:
    import torch
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")
    if cuda_available:
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch not installed; skipping GPU detection.")

In [ ]:
%pip install -q --upgrade pip
%pip install -q pandas numpy scipy matplotlib seaborn scikit-learn pyarrow fastparquet openpyxl google-cloud-storage

In [ ]:
# Core dependencies verification
try:
    import numpy as np
    import pandas as pd
    import scipy
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
    import sklearn
    import pyarrow
    import openpyxl
    print("Core libraries imported successfully.")
except ImportError as e:
    print(f"Libraries not yet installed in current runtime ({e}).")
    print("Run the %pip install cell above when running in Google Colab.")

In [ ]:
from pathlib import Path

# Project root directory setup
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

for p in [DATA_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    print(f"Directory ready: {p.relative_to(BASE_DIR)}")

## 1. Raw Data Storage

The raw data for this project is `attack_dataset.csv`. This csv file (in this repo) will be copied into the notebook and then into the bucket.

#### How I came up with this approach:

- **ephemeral nature of colab runtimes**: Every time you exit your Colab session (or let it go idle), the runtime gets destroyed. The same applies to its local disk space. If all your raw files were stored solely on the Colab instance's disk, they'd get lost after each run. By storing the base file in a GCP bucket, I'm making sure it persists through multiple runs of the notebook.
- **immutability of the raw data**: Per the lectures, raw data should never change. A raw data file could be useful in case a problem comes to light later when cleaning/preprocessing features in a subsequent step; if the raw file has been preserved from the beginning, I'll always have something I can use to rerun everything from a clean slate without worrying about corrupting the data. In my code, I set the local cached version to be read only so I can't accidentally changes it.
- **keeping data out of git** : It is generally considered bad practice to include raw data directly in the Git repository as was shown in lecture 2. The Git repository is meant to track versions of code/configurations/checksums, whereas the raw data goes into the storage system.
- **avoid I/O overhead** : With Lecture 2 in mind, reading lines from the network every time I train an epoch/model or preprocess some subset of a larger dataset can lead to huge I/O overhead and leaves my compute resources unused. By caching `attack_dataset.csv` locally to the NVMe SSD (once) at `/data/raw`, we enable fast sequential access for both Pandas and PyTorch.

In [ ]:
from pathlib import Path
import os
import stat

# Storage configuration
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_FILE = RAW_DATA_DIR / "attack_dataset.csv"

GCS_PROJECT_ID = os.environ.get("GCP_PROJECT", "setu-ai-hdip")
GCS_BUCKET_NAME = "setu-hdip-ai-raw"
GCS_REGION = "europe-west1"
GCS_BLOB_PATH = "cybersecurity/attack_dataset.csv"
REMOTE_GCS_URI = f"gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}"

print(f"Target GCS Bucket: gs://{GCS_BUCKET_NAME}")
print(f"Local Raw Path:    {RAW_DATA_FILE}")

# Spin up GCS databucket and sync raw file
# (uncomment auth in Colab if authenticating for the first time)
# from google.colab import auth
# auth.authenticate_user()

try:
    from google.cloud import storage
    storage_client = storage.Client(project=GCS_PROJECT_ID)
    bucket = storage_client.bucket(GCS_BUCKET_NAME)
    
    if not bucket.exists():
        print(f"Spinning up GCS bucket: gs://{GCS_BUCKET_NAME} ({GCS_REGION})...")
        bucket = storage_client.create_bucket(GCS_BUCKET_NAME, location=GCS_REGION)
        print(f"Bucket ready: gs://{GCS_BUCKET_NAME}")
    else:
        print(f"Connected to existing bucket: gs://{GCS_BUCKET_NAME}")
        
    # Sync to GCS if local file is present
    if RAW_DATA_FILE.exists():
        blob = bucket.blob(GCS_BLOB_PATH)
        if not blob.exists():
            print(f"Uploading {RAW_DATA_FILE.name} to gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}...")
            blob.upload_from_filename(str(RAW_DATA_FILE))
            print("Upload to bucket complete.")
        else:
            print(f"Blob already exists in bucket: gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}")
except ImportError:
    print("Note: google-cloud-storage not installed in current environment.")
except Exception as e:
    print(f"GCS bucket status note: {e}")

# Enforce read-only immutability on local raw file
if RAW_DATA_FILE.exists():
    try:
        os.chmod(RAW_DATA_FILE, stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)
        print("Permissions: Set to read-only (immutable).")
    except Exception as e:
        print(f"Note on permissions: {e}")
else:
    print(f"Local raw file '{RAW_DATA_FILE.name}' not found yet. It can be uploaded or synced in Section 5 (Data Access).")

## 2. Processed Data Storage & File Formats

The location of the processed feature tables and cleaned datasets will follow a two-tier structure based on the source raw data:

- **persistent cloud-based storage object**: `gs://needmyexampleurlhere` stores my data in gcp.
- **fast training access local working cache**: `./data/processed/` on the ssd of the local Colab instance.
- **raw format (CSV)**: the `attack_dataset.CSV` is in CSV format as this is the original format that was retrieveed from the public dataset. CSV is portable and human readable, however, it is extremely inefficient for model training where features need to be iteratively loaded.
- **processed format (Apache Parquet)**: the cleaned features and their respective encoding/scale transformations will be stored in `attack_features_v1.0.parquet`, utilizing the `pyarrow` engine with snappy compresson. (note to self - is parquet needed? must check, i may be misunderstanding it and its functionality)
- **columnar storage (ref slide 21 lecture 2)**: Unlike CSV that stores data row-by-row, Apache Parquet stores data column-by-column. Since most ML feature engineering workflows involve selecting subsets of columns (i.e., numerical vs categorical encoded features), columnar storage means you avoid loading unused columns into RAM.

In [ ]:
from pathlib import Path
import os

# 1. Directory and file paths for processed data
PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PARQUET_FILE = PROCESSED_DATA_DIR / "attack_features_v1.0.parquet"
REMOTE_PROCESSED_URI = "gs://setu-hdip-ai-raw/cybersecurity/processed/attack_features_v1.0.parquet"

print(f"Local processed directory: {PROCESSED_DATA_DIR.relative_to(BASE_DIR)}")
print(f"Local Parquet target:      {PROCESSED_PARQUET_FILE}")
print(f"Remote persistent URI:     {REMOTE_PROCESSED_URI}")

# 2. Helper function to export processed data to Parquet
def export_to_parquet(df, target_path: Path) -> Path:
    """
    Exports processed DataFrame to Apache Parquet format using pyarrow
    and Snappy compression, then prints file size.
    """
    df.to_parquet(target_path, engine="pyarrow", compression="snappy", index=False)
    size_kb = target_path.stat().st_size / 1024
    print(f"Exported Parquet file: {target_path.name} ({size_kb:.2f} KB)")
    return target_path

# Preview conversion if raw data is already cached
if RAW_DATA_FILE.exists():
    try:
        import pandas as pd
        sample_df = pd.read_csv(RAW_DATA_FILE, nrows=50)
        export_to_parquet(sample_df, PROCESSED_PARQUET_FILE)
        print("Sample Parquet conversion verified successfully.")
    except Exception as e:
        print(f"Note on sample export: {e}")
else:
    print("Ready for processed feature exports once raw data is staged.")

## 3. Database / Object Storage Decision

(ref lecture 3 slide 33) i assessed to determine which type of storage architecture was best suited to support this project’s workflow; a Relational Database or object storage with local file system files.

### relational databases (sql - postgresql/mysql)
Relational databases provide the most support for OLTP workloads (i.e., online transaction processing), provide strong consistency guarantees, allow for multiple users to lock rows simultaneously (CRDT like quality) and enable complex join operations (ref Lecture 2 slide 22). Although `attack_dataset.csv` contains data that can be organized into a table format, all aspects of the model training pipeline used by this project will involve offline batch training. Each iteration of the training loop reads through the input data sequentially in mini batches (thus there are no need for random row lookups (`where id=x`) as this wouldnt be suitable for this projects needs and goals). Therefore, the addition of an external sql server would impose additional network latency and serialization overhead each time training occurred (the i/o bottleneck described in Lecture 2 slide 8) as well as require additional server configuration and expense to host. Realistically, a relational database is not needed for this project.

### NoSQL/documents stores (mongodb/cassandra)
While NoSQL systems excel at handling large amounts of writes in a distributed environment, managing variable/nested Document structures, or supporting real-time events; they don't provide significant advantages over relational databases for projects that have fixed schema tables. In the case of this dataset, since it is stored in a fixed table format, storing the dataset as JSON/BSON documents in mongodb would result in only increased CPU usage during deserialization with little added benefit. Using somehting like mongodb would also require more configuration.

### selected Approach: cloud object storage + local NVMe ssd caching
In place of a database, this project uses a combination of cloud object storage (I'm using a bucket on gcp) and local ssd caching:

- **cloud object storage (google cloud storage / google drive):** provides a durable data lake for both raw and processed datasets (lecture 3 slide 5); provides high durability at low cost per gb without the need to maintain a database.
- **local NVMe ssd cache:** during active training sessions within Colab, the compressed Parquet files are read directly from the local drive. By doing so, we maximize the sequential read of our CPU/GPU while eliminating any potential network bottlenecks completely.

Given that this is an offline machine learning project on a static dataset; using object storage along with local Parquet files is significantly less expensive, easier to manage and faster than running a database.

In [ ]:
# Storage architecture summary check
storage_decision = {
    "workload_type": "Offline Batch Machine Learning",
    "access_pattern": "Sequential mini-batch reads",
    "selected_primary_storage": "Cloud Object Storage (GCS / Google Drive)",
    "selected_runtime_cache": "Local NVMe SSD File System",
    "selected_processed_format": "Apache Parquet (Snappy compressed)",
    "database_required": False,
    "justification": "Avoids DB connection/serialization latency; maximizes sequential I/O throughput."
}

print("Storage Architecture Decision Summary:")
for k, v in storage_decision.items():
    print(f"  {k:27s}: {v}")

## 4. Data Versioning



In [ ]:
from pathlib import Path
import hashlib
import json
import time

DATASET_VERSION = "v1.0"

# Compute SHA-256 checksum and generate version manifest
if RAW_DATA_FILE.exists():
    hasher = hashlib.sha256()
    with open(RAW_DATA_FILE, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    sha256_hash = hasher.hexdigest()
    file_size_kb = RAW_DATA_FILE.stat().st_size / 1024
    
    version_metadata = {
        "dataset_name": "attack_dataset.csv",
        "version": DATASET_VERSION,
        "sha256": sha256_hash,
        "size_kb": round(file_size_kb, 2),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "storage_uri": REMOTE_GCS_URI
    }
    
    manifest_file = RAW_DATA_DIR / "dataset_manifest.json"
    with open(manifest_file, "w") as f:
        json.dump(version_metadata, f, indent=2)
        
    print(f"Dataset Version: {DATASET_VERSION}")
    print(f"SHA-256 Hash:    {sha256_hash}")
    print(f"File Size:       {file_size_kb:.2f} KB")
    print(f"Version Manifest saved to: {manifest_file.relative_to(BASE_DIR)}")
else:
    print(f"Note: '{RAW_DATA_FILE.name}' not found yet. Run Section 5 (Data Access) to upload or sync the dataset, then re-run this cell.")

## 5. Data Access



In [ ]:
from pathlib import Path
import os
import shutil

# 1. Google Drive mount (uncomment in Colab if using Drive persistence)
# from google.colab import drive
# drive.mount('/content/drive')

GDRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/HDip_AI_Project")
GDRIVE_RAW_FILE = GDRIVE_PROJECT_DIR / "data/raw/attack_dataset.csv"

# 2. Check Drive mount first; sync to local cache if present
if GDRIVE_RAW_FILE.exists() and not RAW_DATA_FILE.exists():
    shutil.copy(GDRIVE_RAW_FILE, RAW_DATA_FILE)
    print(f"Synced raw data from Drive to local cache: {RAW_DATA_FILE}")

# 3. If still missing, prompt user for CSV file upload
if not RAW_DATA_FILE.exists():
    try:
        from google.colab import files
        print("Please select and upload your 'attack_dataset.csv' file:")
        uploaded = files.upload()
        for fname, data in uploaded.items():
            dest = RAW_DATA_DIR / fname
            with open(dest, "wb") as f:
                f.write(data)
            print(f"Uploaded and saved: {dest}")
            if fname != RAW_DATA_FILE.name and not RAW_DATA_FILE.exists():
                shutil.copy(dest, RAW_DATA_FILE)
    except ImportError:
        # Fallback for local run outside Colab
        print(f"Running outside Colab. '{RAW_DATA_FILE.name}' not found in {RAW_DATA_DIR}.")
        print("Place the CSV file into './data/raw/' to proceed.")
else:
    print(f"Raw data available for access at: {RAW_DATA_FILE}")

## 6. Data Split / Validation Strategy

Train/dev/test partitioning or cross-validation strategy, avoiding data leakage.

## 7. Feature Description

Key features used from `attack_dataset.csv` and what each feature represents.

## 8. Data Types and Formats

Breakdown of data types (numerical, categorical, timestamps) and schema handling.

## 9. Reproducibility of Data Collection

Data sources, collection procedures, and parameters needed to reproduce the dataset.

## 10. Reproducibility of Preprocessing

Detailed steps to clean, filter, and transform the raw data into processed features.

## Verification & Baseline Pipeline Test

This should be checks to make sure everything is spun up properly.